# Causal BC on AntMaze Large

In [1]:
import random
import torch
import pickle
import os
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict

from causal_gym import AntMazePCH
from causal_rl.algo.imitation.imitate import *
from causal_rl.algo.imitation.finetune import *

<frozen importlib._bootstrap>:241: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.
/home/et2842/miniconda3/envs/causalenv/lib/python3.11/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [3]:
num_steps = 1000
seed = 0
lookback = 100
hidden_dims = {'O'}

random.seed(seed)
torch.manual_seed(seed)

In [4]:
# for training: regular W, O hidden
train_env = AntMazePCH(env_id='antmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=True, custom_hidden=hidden_dims, seed=seed)

# for eval: corrupted W, O hidden
eval_env = AntMazePCH(env_id='antmaze-large-navigate-singletask-task1-v0', num_steps=num_steps, expert_mode=False, seed=seed)

## Causal Graph Analysis

In [5]:
# to save time; conceptually the same
small_steps = lookback + 1
small_env = AntMazePCH(env_id='antmaze-large-navigate-singletask-task1-v0', num_steps=small_steps, seed=seed)
G = parse_graph(small_env.get_graph)
X_small = {f'X{t}' for t in range(small_steps)}
Y = f'Y{small_steps}'

X = {f'X{t}' for t in range(num_steps)}
obs_prefix = train_env.env.observed_unobserved_vars[0]

In [6]:
Z_sets = find_sequential_pi_backdoor(G, X_small, Y, obs_prefix)

base_step = small_steps - 1
base_Z_set = Z_sets[f'X{base_step}']

for i in range(base_step + 1, num_steps):
    updated_base_Z_set = set()
    for v in base_Z_set:
        updated_base_Z_set.add(f'{v[0]}{int(v[1:]) + i - lookback}')

    Z_sets[f'X{i}'] = updated_base_Z_set

Z_sets['X1']

{'A0', 'A1', 'J0', 'J1', 'L0', 'L1', 'P0', 'P1', 'T0', 'T1', 'X0'}

## Expert Trajectories

In [7]:
with open('/home/et2842/causal/expert_traj_antlarge.pkl', 'rb') as f:
    records = pickle.load(f)

print(f'loaded {len(records)} trajectories')

loaded 400026 trajectories


In [8]:
dims = {
    'P': 3,
    # 'O': 4,
    'A': 8,
    'L': 3,
    'T': 3,
    'J': 8,
    'W': 2,
    'X': 8,
}

## Training

In [9]:
hidden_size = 256
lr = 3e-4
batch_size = 2048
patience = 15
num_blocks = 4
epochs = 100
dropout = 0.0

In [10]:
cbc_model, cbc_slots, cbc_Z_trim = train_single_policy_long_horizon(
    records,
    Z_sets,
    dims=dims,
    epochs=epochs,
    include_vars=obs_prefix,
    lookback=lookback,
    continuous=True,
    num_actions=train_env.action_space.shape[0],
    hidden_dim=hidden_size,
    num_blocks=num_blocks,
    dropout=dropout,
    lr=lr,
    batch_size=batch_size,
    patience=patience,
    device=device,
    seed=seed,
    action_bounds=(train_env.action_space.low, train_env.action_space.high)
)

cbc_policy = shared_policy_fn_long_horizon(cbc_model, cbc_slots, cbc_Z_trim, continuous=True, device=device)
cbc_policies = make_shared_policy_dict(cbc_policy)

[LongHorizon] Epoch 1: train loss = 0.119412, val loss = 0.083212.


[LongHorizon] Epoch 2: train loss = 0.070291, val loss = 0.062963.


[LongHorizon] Epoch 3: train loss = 0.055760, val loss = 0.053389.


[LongHorizon] Epoch 4: train loss = 0.047079, val loss = 0.050534.


[LongHorizon] Epoch 5: train loss = 0.041731, val loss = 0.044078.


...

[LongHorizon] Epoch 96: train loss = 0.004622, val loss = 0.021607.


[LongHorizon] Epoch 97: train loss = 0.004315, val loss = 0.021853.


[LongHorizon] Epoch 98: train loss = 0.004309, val loss = 0.021454.


[LongHorizon] Epoch 99: train loss = 0.004223, val loss = 0.021432.


[LongHorizon] Epoch 100: train loss = 0.004307, val loss = 0.021753.


## Evaluation

In [11]:
num_eval_eps = 10
cbc_returns = collect_imitator_trajectories(
    env=eval_env,
    policies=cbc_policies,
    num_episodes=num_eval_eps,
    max_steps=num_steps,
    hidden_dims=hidden_dims,
    show_progress=True,
    seed=seed + 90210,
)

len(cbc_returns)

Starting episode 1/10...


  Episode 1 ended at step 1000 (terminated: False, truncated: True).
Starting episode 2/10...


  Episode 2 ended at step 1000 (terminated: False, truncated: True).
Starting episode 3/10...


  Episode 3 ended at step 1000 (terminated: False, truncated: True).
Starting episode 4/10...


  Episode 4 ended at step 1000 (terminated: False, truncated: True).
Starting episode 5/10...


...

  Episode 7 ended at step 563 (terminated: True, truncated: False).
Starting episode 8/10...


  Episode 8 ended at step 564 (terminated: True, truncated: False).
Starting episode 9/10...


  Episode 9 ended at step 576 (terminated: True, truncated: False).
Starting episode 10/10...


  Episode 10 ended at step 1000 (terminated: False, truncated: True).
Finished collecting imitator trajectories.


8662

In [12]:
cbc_episode_rewards = defaultdict(float)
for rec in cbc_returns:
    ep = rec['episode']
    cbc_episode_rewards[ep] += float(rec['reward'])

cbc_rewards = [cbc_episode_rewards[e] for e in range(num_eval_eps)]
sum(cbc_rewards) / num_eval_eps

-289.84414703192635

## Save Model

In [13]:
SAVE_DIR = '/home/et2842/causal/causalrl/models'
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_PATH = os.path.join(SAVE_DIR, 'cbc_k100_antlarge.pt')

checkpoint = {
    "state_dict": cbc_model.state_dict(),
    "slots": cbc_slots,
    "Z_trim": cbc_Z_trim,
    "dims": dims,
    "lookback": lookback,
    "continuous": True,
    "num_actions": train_env.action_space.shape[0],
    "hidden_dim": hidden_size,
    "num_blocks": num_blocks,
    "dropout": dropout,
    "layernorm": True,
    "final_tanh": True,
    "action_bounds_low": eval_env.action_space.low,
    "action_bounds_high": eval_env.action_space.high,
    "input_dim": int(cbc_model.hidden.in_features),
}

torch.save(checkpoint, MODEL_PATH)
print(f'Saved to: {MODEL_PATH}')

Saved to: /home/et2842/causal/causalrl/models/cbc_k100_antlarge.pt
